## Optuna practise

In [1]:
# import necessary linararies
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#load the pima indian dataset from sklearn
#NBote:-Scikit learn built in "load_diabetes"is a regression model
#we will load the actual diabetets from an external source

import pandas as pd
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
df=pd.read_csv(url,names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [3]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [5]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-11-21 08:41:51,861] A new study created in memory with name: no-name-fdbea03d-6c82-4d9b-9acd-723989c47e4a
[I 2025-11-21 08:41:52,992] Trial 0 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 155, 'max_depth': 3}. Best is trial 0 with value: 0.7541899441340782.
[I 2025-11-21 08:41:54,110] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 162, 'max_depth': 5}. Best is trial 1 with value: 0.7672253258845437.
[I 2025-11-21 08:41:54,906] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 143, 'max_depth': 13}. Best is trial 2 with value: 0.7690875232774674.
[I 2025-11-21 08:41:55,351] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 85, 'max_depth': 5}. Best is trial 2 with value: 0.7690875232774674.
[I 2025-11-21 08:41:56,272] Trial 4 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 191, 'max_depth': 12}. Best is trial 4 with value: 0.77467411

In [6]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7839851024208566
Best hyperparameters: {'n_estimators': 119, 'max_depth': 15}


In [7]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


## sampler in Optuna

## for random serach CV

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [9]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-11-21 08:45:09,203] A new study created in memory with name: no-name-7e7bcf05-779d-41e2-9813-9c42bedbb2ca
[I 2025-11-21 08:45:09,871] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 78, 'max_depth': 17}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-11-21 08:45:10,521] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 84, 'max_depth': 9}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-11-21 08:45:11,623] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 164, 'max_depth': 6}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-11-21 08:45:12,478] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 160, 'max_depth': 13}. Best is trial 3 with value: 0.7690875232774674.
[I 2025-11-21 08:45:13,056] Trial 4 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 125, 'max_depth': 7}. Best is trial 4 with value: 0.778398510

In [10]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 121, 'max_depth': 20}


##  For Grid Search CV

In [11]:
# we explctly define searh_space in gridsearchcv
search_space={
    'n_estimators':[50,100,150,200],
    'max_depth':[5,10,15,20]
}

In [12]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-11-21 08:46:10,169] A new study created in memory with name: no-name-971ca632-6865-4144-81ae-4e1286337d70
[I 2025-11-21 08:46:10,907] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-11-21 08:46:11,834] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-11-21 08:46:12,116] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-11-21 08:46:12,590] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-11-21 08:46:13,081] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [13]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


## Optuna Visualization

In [14]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [15]:
#Optimization HIstory
plot_optimization_history(study).show()

In [16]:
# parrallel coodrinate chord
plot_parallel_coordinate(study).show()

In [17]:
#3 sl,ice plot
plot_slice(study).show()

In [18]:
#4 plot contour Plot
plot_contour(study).show()

In [19]:
#5 Hyparameter Importance
plot_param_importances(study).show()

In [20]:
#best thing inoptine
# define by run
# Dynamic Search Species

### Optimizing Multiple ML Models

In [21]:
#library
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC

In [22]:

#define the obbjection functon for the optima
def objective(trial):
    #choose the algorithm to tune
    classifier_name=trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting'])
    
    if classifier_name == "SVM":
        #SVM Hyperparameter
        c=trial.suggest_float('C',0.1,100,log=True)
        kernel=trial.suggest_categorical('kernel',['linear','rbf','poly','sigmoid'])
        gamma=trial.suggest_categorical('gamma',['scale','auto'])
        model=SVC(C=c,kernel=kernel,gamma=gamma,random_state=42)
        
    elif classifier_name == "RandomForest":
        #random forst hypermeters
        n_estimators=trial.suggest_int('n_estimators',50,300)
        max_depth=trial.suggest_int('max_depth',3,20)
        min_samples_split=trial.suggest_int('min_samples_split',2,10)
        min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10)
        bootstrap=trial.suggest_categorical('bootstrap',[True,False])
        
        model=RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )
    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        
    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [23]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100) 
#  default tpe sample

[I 2025-11-21 08:46:23,525] A new study created in memory with name: no-name-12d26751-34a2-43b3-af34-3b269ad95630
[I 2025-11-21 08:46:23,555] Trial 0 finished with value: 0.7318435754189944 and parameters: {'classifier': 'SVM', 'C': 1.55776184714392, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7318435754189944.
[I 2025-11-21 08:46:23,583] Trial 1 finished with value: 0.7672253258845437 and parameters: {'classifier': 'SVM', 'C': 0.7825676775783441, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 1 with value: 0.7672253258845437.
[I 2025-11-21 08:46:24,869] Trial 2 finished with value: 0.7523277467411545 and parameters: {'classifier': 'RandomForest', 'n_estimators': 271, 'max_depth': 13, 'min_samples_split': 4, 'min_samples_leaf': 9, 'bootstrap': True}. Best is trial 1 with value: 0.7672253258845437.
[I 2025-11-21 08:46:25,711] Trial 3 finished with value: 0.7672253258845437 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 157, 'learning_rate':

In [24]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1164783426454297, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [25]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.731844,2025-11-21 08:46:23.526879,2025-11-21 08:46:23.555706,0 days 00:00:00.028827,1.557762,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.767225,2025-11-21 08:46:23.557257,2025-11-21 08:46:23.583622,0 days 00:00:00.026365,0.782568,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.752328,2025-11-21 08:46:23.583622,2025-11-21 08:46:24.869528,0 days 00:00:01.285906,NaN,True,RandomForest,NaN,NaN,NaN,13.0,9.0,4.0,271.0,COMPLETE
3,3,0.767225,2025-11-21 08:46:24.870543,2025-11-21 08:46:25.711252,0 days 00:00:00.840709,NaN,NaN,GradientBoosting,NaN,NaN,0.036335,4.0,8.0,10.0,157.0,COMPLETE
4,4,0.729981,2025-11-21 08:46:25.713233,2025-11-21 08:46:27.832308,0 days 00:00:02.119075,NaN,NaN,GradientBoosting,NaN,NaN,0.099532,18.0,8.0,9.0,195.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2025-11-21 08:46:46.911145,2025-11-21 08:46:46.935563,0 days 00:00:00.024418,0.115858,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.772812,2025-11-21 08:46:46.936573,2025-11-21 08:46:47.550409,0 days 00:00:00.613836,NaN,False,RandomForest,NaN,NaN,NaN,15.0,7.0,5.0,164.0,COMPLETE
97,97,0.787709,2025-11-21 08:46:47.551393,2025-11-21 08:46:47.577881,0 days 00:00:00.026488,0.174376,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.744879,2025-11-21 08:46:47.578904,2025-11-21 08:46:49.341869,0 days 00:00:01.762965,NaN,NaN,GradientBoosting,NaN,NaN,0.159095,6.0,4.0,8.0,212.0,COMPLETE


In [26]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 79
GradientBoosting    11
RandomForest        10
Name: count, dtype: int64

In [27]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()


params_classifier
GradientBoosting    0.751820
RandomForest        0.764618
SVM                 0.778304
Name: value, dtype: float64